# Move Semantics (Semântica de Movimento) em C++20 — Alakoro FiberSense

**Objetivo:** entender como a semântica de movimento do C++20 evita cópias caras de arrays DAS/DTS/DSS no motor de inferência do Alakoro FiberSense.

**Público-alvo:** engenheiros que conhecem Python e querem entender por que o C++20 do Alakoro é *move-only* em pontos críticos.

**Arquivos-fonte consultados:**
- `src/cpp/include/alakoro/inference_engine.hpp` — `InferenceResult`, `ResultGenerator`, `collect_results`, `execute_rule`, regras canônicas.
- `src/cpp/src/bindings.cpp` — bindings pybind11, `vector_to_numpy`.
- `tests/test_cpp_core.py` — testes da camada nativa.

> **Nota sobre células C++:** este notebook roda em um kernel Python. Os blocos `cpp` são apenas ilustrativos (markdown). Compile-os separadamente com um compilador C++20 (g++13+, clang++16+).

## 1. O problema: cópias caras de objetos grandes

No processamento de dados de fibra óptica (DAS/DTS/DSS) do Alakoro, trabalhamos com volumes significativos: trilhas de amplitude/strain ao longo de milhares de amostras temporais e centenas ou milhares de canais de profundidade. Um único `std::vector<double>` que represente uma aquisição pode facilmente ocupar dezenas de megabytes.

Quando C++ copia esses objetos — seja um vetor de dados brutos, uma matriz de resultados ou uma corrotina que encapsula estado de processamento — o custo é proporcional ao tamanho do buffer. Para o motor de inferência (`InferenceEngine`), isso significa cópias indesejáveis entre:

- a extração de perfis médios (`temporal_mean`);
- a remoção de baseline (`remove_polynomial_baseline`, `remove_median_baseline`);
- a acumulação de `InferenceResult`;
- o retorno de resultados para Python via pybind11.

A **semântica de movimento** (move semantics), introduzida no C++11 e aperfeiçoada no C++14/C++17/C++20, resolve exatamente esse problema: em vez de duplicar recursos, transferimos a posse (*ownership*) dos dados de um objeto para outro. O buffer de memória não é copiado; apenas os ponteiros internos são trocados.

In [ ]:
# Experimento Python: atribuicao nao copia; deepcopy copia.
import copy
import numpy as np

a = [1.0, 2.0, 3.0]
b = a              # b aponta para o MESMO objeto
b.append(4.0)
print("a =", a)    # [1.0, 2.0, 3.0, 4.0] — Python eh referencia

c = copy.deepcopy(a)
c.append(5.0)
print("a =", a)
print("c =", c)

# NumPy: view vs copy
x = np.arange(1_000_000, dtype=np.float64)
y = x              # view/referencia, zero-copy
z = x.copy()       # copia real na memoria
print("x data ptr:", x.ctypes.data)
print("y data ptr:", y.ctypes.data)  # igual a x
print("z data ptr:", z.ctypes.data)  # diferente

## 2. Conceitos fundamentais

### 2.1 lvalue vs rvalue

- **lvalue** (*left value*): expressao que tem identidade e endereco. Pode aparecer a esquerda de uma atribuicao.
  - Exemplos: uma variavel nomeada, um elemento de vetor, um retorno por referencia.
- **rvalue** (*right value*): expressao temporaria, sem identidade propria. Geralmente aparece a direita de uma atribuicao.
  - Exemplos: o resultado de uma funcao que retorna por valor, um literal, `std::move(x)`.

```cpp
std::vector<double> make_data();          // retorna um rvalue
std::vector<double> v = make_data();      // v eh um lvalue
v[0] = 1.0;                               // v[0] eh um lvalue
std::vector<double> w = std::move(v);     // std::move(v) eh um rvalue
```

### 2.2 Referencias

| Sintaxe | Significado | Quando usar |
|---------|-------------|-------------|
| `T&` | Referencia nao-constante para lvalue | Modificar o objeto original sem copia-lo |
| `const T&` | Referencia constante para lvalue/rvalue | Ler o objeto sem copia-lo; liga-se a temporarios |
| `T&&` | Referencia para rvalue | Implementar move semantics; "roubar" recursos de um temporario |

> A regra pratica: use `const T&` para leitura, `T&&` para mover, e `T&` quando precisar modificar o objeto original.

### 2.3 Move constructor e move assignment operator

Todo tipo pode ter, alem do construtor de copia e do operador de atribuicao por copia, versoes de **movimento**:

```cpp
class Buffer {
public:
    // Construtor padrao
    Buffer() = default;

    // Construtor de copia (deep copy)
    Buffer(const Buffer& other) : data_(other.data_) { /* copia elementos */ }

    // Operador de atribuicao por copia
    Buffer& operator=(const Buffer& other) {
        if (this != &other) {
            data_ = other.data_;  // copia
        }
        return *this;
    }

    // Move constructor: rouba os recursos de other
    Buffer(Buffer&& other) noexcept : data_(std::move(other.data_)) {
        // other eh deixado em estado valido, mas vazio
    }

    // Move assignment operator
    Buffer& operator=(Buffer&& other) noexcept {
        if (this != &other) {
            data_ = std::move(other.data_);  // move o vetor interno
        }
        return *this;
    }

private:
    std::vector<double> data_;
};
```

A palavra-chave `noexcept` eh importante: se um move constructor lancar excecao, algumas operacoes da biblioteca padrao (como realocacao de vetores) recuam para copias, perdendo o ganho de performance.

### 2.4 `std::move`: um cast, nao uma operacao de movimento

`std::move(x)` **nao move** `x`. Ele apenas converte `x` em um rvalue, permitindo que o compilador escolha a sobrecarga de movimento.

```cpp
std::vector<double> a = {1.0, 2.0, 3.0};
std::vector<double> b = std::move(a);  // agora b tem os dados; a esta vazio
```

Apos `std::move(a)`, o estado de `a` eh valido mas indefinido em termos de conteudo. A unica operacao segura garantida eh destrui-lo ou reassinar.

### 2.5 Rule of Zero / Rule of Five

- **Rule of Zero**: se seus membros gerenciam recursos corretamente (como `std::vector`, `std::string`, `std::unique_ptr`), voce **nao precisa** escrever destrutor, copia ou movimento. O compilador gera tudo corretamente.
- **Rule of Five**: se voce precisa implementar **qualquer um** de (destrutor, copy constructor, copy assignment, move constructor, move assignment), provavelmente precisa implementar os cinco — ou declarar explicitamente o que nao deve existir (`= delete`).

No Alakoro, `ResultGenerator` segue a **Rule of Five** de proposito: ele possui um `std::coroutine_handle`, que eh um recurso unico. Por isso, a copia eh deletada e o movimento eh implementado manualmente.

In [ ]:
# Python nao tem lvalue/rvalue nem std::move. Toda atribuicao eh referencia.
a = [1, 2, 3]
b = a
print(a is b)  # True — nenhuma copia

# Para "simular" uma transferencia de posse, podemos apenas apagar a referencia antiga
def move_list(src):
    """Remove a variavel src do escopo global/local, entregando o objeto."""
    return src

a = [1, 2, 3]
b = move_list(a)
del a  # a nao existe mais; b eh a unica referencia
print(b)

# Mas ainda eh o mesmo objeto: sem equivalente direto ao std::move de C++.
print(type(b), len(b))

## 3. Exemplos minimos: antes e depois do move

### 3.1 Antes: copia explicita de vetores

```cpp
std::vector<double> temporal_mean(const std::vector<double>& data) {
    std::vector<double> mean(data.size());
    // ... preenche mean ...
    return mean;  // NRVO/Move ja ajudam, mas vamos supor o pior caso
}

auto profile = temporal_mean(raw);
auto anomaly = remove_polynomial_baseline(profile, 2);  // copia profile!
```

Se `profile` tiver 100 mil elementos, a chamada acima pode copiar 800 KB desnecessariamente.

### 3.2 Depois: move em vez de copiar

```cpp
inline std::vector<double> remove_polynomial_baseline(std::vector<double> profile,
                                                       std::size_t degree = 2) {
    // recebemos por valor para poder trabalhar no proprio buffer
    if (profile.size() < degree + 2) return profile;
    const std::size_t n = profile.size();
    // ... usa profile diretamente para calcular baseline e subtrai ...
    return profile;  // move out (NRVO ou move)
}

auto mean_profile = detail::temporal_mean(dts, n_times, n_channels);
auto anomaly = detail::remove_polynomial_baseline(std::move(mean_profile), 2);
```

Aqui, `mean_profile` eh **movido** para dentro de `remove_polynomial_baseline`. Nenhuma alocacao extra ocorre para o vetor de anomalia: o mesmo buffer eh reutilizado.

Esse padrao aparece repetidamente no `inference_engine.hpp`, por exemplo em `SlopeVelocityRule`:

```cpp
auto first_half = detail::temporal_mean(dts.subspan(0, mid_t * n_channels),
                                         mid_t, n_channels);
auto second_half = detail::temporal_mean(
    dts.subspan(mid_t * n_channels, (n_times - mid_t) * n_channels),
    n_times - mid_t, n_channels);

auto anom1 = detail::remove_polynomial_baseline(std::move(first_half), 2);
auto anom2 = detail::remove_polynomial_baseline(std::move(second_half), 2);
```

In [ ]:
# Benchmark Python vs C++-like copy em arrays grandes
import numpy as np
import time

n = 10_000_000
profile = np.random.rand(n)

# Simula "copia" (sem move): aloca novo array
def baseline_copy(p):
    out = p.copy()  # copia real
    out -= out.mean()
    return out

t0 = time.perf_counter()
_ = baseline_copy(profile)
t_copy = time.perf_counter() - t0
print(f"Com copia: {t_copy*1000:.2f} ms, novo array alocado")

# Simula "move": reutiliza o buffer in-place
def baseline_move(p):
    p -= p.mean()  # modifica p in-place
    return p

t0 = time.perf_counter()
_ = baseline_move(profile)
t_move = time.perf_counter() - t0
print(f"In-place (estilo move): {t_move*1000:.2f} ms, buffer reutilizado")
print(f"Speedup: {t_copy/t_move:.1f}x")

# Observacao: em C++ a vantagem eh ainda maior porque evitamos copia + alocacao
# e ainda podemos usar o buffer morto da variavel original.

## 4. Aplicacao no Alakoro

### 4.1 `ResultGenerator`: recurso unico, delete copy, implement move

A `ResultGenerator` eh a corrotina que produz `InferenceResult` via `co_yield`. Por baixo, ela guarda um `std::coroutine_handle<promise_type>`, um identificador de frame de corrotina — um recurso que nao pode ser compartilhado.

No arquivo `src/cpp/include/alakoro/inference_engine.hpp` (linhas 228–281):

```cpp
struct ResultGenerator {
    struct promise_type {
        InferenceResult current_value;

        ResultGenerator get_return_object() {
            return ResultGenerator{std::coroutine_handle<promise_type>::from_promise(*this)};
        }

        std::suspend_always initial_suspend() noexcept { return {}; }
        std::suspend_always final_suspend() noexcept { return {}; }
        void unhandled_exception() { std::terminate(); }
        void return_void() noexcept {}

        std::suspend_always yield_value(InferenceResult value) noexcept {
            current_value = std::move(value);  // move o resultado do co_yield
            return {};
        }
    };

    using handle_type = std::coroutine_handle<promise_type>;

    explicit ResultGenerator(handle_type h) : handle_(h) {}

    // Nao copiavel: coroutine_handle eh um recurso unico.
    ResultGenerator(const ResultGenerator&) = delete;
    ResultGenerator& operator=(const ResultGenerator&) = delete;

    // Move constructor: transfere posse do handle
    ResultGenerator(ResultGenerator&& other) noexcept : handle_(other.handle_) {
        other.handle_ = nullptr;
    }

    // Move assignment: destroi handle antigo, rouba o novo
    ResultGenerator& operator=(ResultGenerator&& other) noexcept {
        if (this != &other) {
            if (handle_) handle_.destroy();
            handle_ = other.handle_;
            other.handle_ = nullptr;
        }
        return *this;
    }

    ~ResultGenerator() {
        if (handle_) handle_.destroy();
    }

    bool done() const noexcept { return handle_.done(); }
    void resume() { if (handle_) handle_.resume(); }

    const InferenceResult& value() const noexcept {
        return handle_.promise().current_value;
    }

private:
    handle_type handle_;
};
```

Observe:

1. **Copia deletada**: `ResultGenerator(const ResultGenerator&) = delete;`. Tentar copiar uma corrotina eh um erro de compilacao, garantindo que nunca havera dois handles para o mesmo frame.
2. **Move constructor `noexcept`**: rouba o `handle_` e zera o objeto fonte.
3. **Move assignment com self-check**: destroi o recurso antigo antes de assumir o novo, evitando vazamento.
4. **yield com move**: `current_value = std::move(value);` evita copiar `std::string`s dentro de `InferenceResult`.

In [ ]:
# Equivalente Python conceitual: um gerador tambem eh "move-only" no sentido
# de que nao faz sentido copiar seu estado interno.

def result_generator():
    yield {"event": "joule_thomson", "confidence": 0.9}
    yield {"event": "warm_back", "confidence": 0.7}

gen = result_generator()
print(next(gen))
print(next(gen))

# Nao existe "copiar" um gerador em Python — voce pode apenas criar outro.
gen2 = result_generator()
print("novo gerador:", next(gen2))

# Em C++, std::move(gen) transferiria o handle interno; em Python, reassinamos
# a variavel para indicar que o objeto original nao deve mais ser usado.
gen3 = gen
del gen  # move semantico: remove a referencia original
print("ainda acessivel via gen3?", gen3)

### 4.2 `collect_results` e consumo da corrotina

A funcao `collect_results` eh a ponte entre a API corrotinada interna e a API sincrona exposta. No codigo atual, ela recebe `ResultGenerator` por valor (linha 1230):

```cpp
inline std::vector<InferenceResult> collect_results(ResultGenerator gen) {
    std::vector<InferenceResult> results;
    results.reserve(4);
    while (!gen.done()) {
        gen.resume();
        if (!gen.done()) {
            results.push_back(std::move(gen.value()));  // move do generator para o vetor
        }
    }
    return results;
}
```

Receber por valor ainda permite que o caller mova a corrotina no momento da chamada:

```cpp
auto partial = collect_results(std::move(gen));
```

Uma alternativa ainda mais explicita, que deixa claro que a funcao **consome** o gerador, seria usar uma rvalue reference:

```cpp
inline std::vector<InferenceResult> collect_results(ResultGenerator&& gen) {
    std::vector<InferenceResult> results;
    results.reserve(4);
    while (!gen.done()) {
        gen.resume();
        if (!gen.done()) {
            results.push_back(std::move(gen.value()));
        }
    }
    return results;
}
```

Com essa assinatura, `collect_results(std::move(gen))` continua valido, mas `collect_results(gen)` se torna um erro de compilacao, aumentando a seguranca: ninguem pode esquecer de mover acidentalmente e depois tentar reusar `gen`.

In [ ]:
# Python: consumir um gerador e mover resultados para uma lista
def collect_results(gen):
    results = []
    for value in gen:
        results.append(value)  # em Python, dicts sao referencias; nao ha copia estrutural
    return results

g = result_generator()
partial = collect_results(g)
print(partial)

# Apos consumir, o gerador esgotou
try:
    next(g)
except StopIteration:
    print("Gerador esgotado — equivalente a usar o ResultGenerator movido no C++.")

### 4.3 `execute_rule`: movendo o generator e os resultados parciais

No `InferenceEngine`, cada regra eh executada dentro de `execute_rule`, e seus resultados sao fundidos no vetor final via `std::make_move_iterator` (linhas 1276–1316):

```cpp
template <CanonicalEvent E>
void execute_rule(std::span<const double> dts,
                  std::span<const double> das,
                  std::size_t n_times,
                  std::size_t n_channels,
                  const InferenceMetadata& meta,
                  std::vector<InferenceResult>& out) const {
    ResultGenerator gen = [&]() {
        if constexpr (E == CanonicalEvent::JouleThomson) {
            return JouleThomsonRule::apply(dts, das, n_times, n_channels, meta);
        }
        // ... outras regras ...
    }();

    auto partial = collect_results(std::move(gen));  // move a corrotina

    out.insert(out.end(),
               std::make_move_iterator(partial.begin()),
               std::make_move_iterator(partial.end()));  // move cada InferenceResult
}
```

Sem `std::move(gen)`, `collect_results` tentaria copiar a corrotina — o que falharia em compilacao, gracas ao `= delete` na copia. Sem `std::make_move_iterator`, os `InferenceResult` (que contem varias `std::string`) seriam copiados um a um para `out`.

In [ ]:
# Python: fundir listas de dicionarios (sem copia estrutural)
out = []
partial = [{"event": "a"}, {"event": "b"}]

# Em Python, extend copia as referencias, nao os objetos.
out.extend(partial)
print(out)
print(out[0] is partial[0])  # True — referencias

# Se quisermos realmente uma copia independente:
import copy
out2 = []
out2.extend(copy.deepcopy(p) for p in partial)
print(out2[0] is partial[0])  # False

### 4.4 pybind11: `vector_to_numpy` e `py::return_value_policy::move`

No `src/cpp/src/bindings.cpp`, a camada Python recebe os vetores de resultados por valor e os converte para listas/tuplas de NumPy. A funcao `vector_to_numpy` (linhas 43–50) ja aproveita a semantica de movimento internamente:

```cpp
template <typename T>
py::array_t<T> vector_to_numpy(std::vector<T> vec) {
    const std::size_t size = vec.size();
    T* data = vec.data();
    auto* owned = new std::vector<T>(std::move(vec));  // move o vetor para a capsula
    py::capsule capsule(owned, [](void* p) { delete static_cast<std::vector<T>*>(p); });
    return py::array_t<T>({size}, {sizeof(T)}, data, capsule);
}
```

Aqui, o `std::vector<T>` eh movido para dentro de um `std::vector<T>` alocado no heap, que vive dentro de uma `py::capsule`. O array NumPy aponta para os mesmos dados sem copia.

Quando registramos funcoes que retornam objetos C++ diretamente (nao vetores convertidos manualmente), podemos instruir o pybind11 a mover o retorno em vez de copiar:

```cpp
m.def("make_inference_result",
      []() { return InferenceResult{...}; },
      py::return_value_policy::move,
      "Retorna um InferenceResult movido para Python");
```

No binding atual do `infer_events_d`, o retorno eh `std::vector<InferenceResult>`. Como `pybind11/stl.h` esta incluido, o pybind11 converte automaticamente o vetor para uma `list` Python. Internamente, cada `InferenceResult` eh **copiado** durante essa conversao, porque o padrao para tipos registrados eh `copy` quando expostos dentro de containers STL.

Se quisermos garantir movimento tambem no retorno para Python, a alternativa eh:

1. Retornar por valor e confiar na otimizacao de copia (RVO/NRVO) do C++ no lado nativo.
2. Para objetos pesados, expor explicitamente o move constructor no binding:

```cpp
py::class_<InferenceResult>(m, "InferenceResult")
    .def(py::init<>())
    .def(py::init<const InferenceResult&>())  // copy
    .def(py::init<InferenceResult&&>())       // move (geralmente implicito)
    // ... membros ...
```

Para o caso do Alakoro, a copia de `InferenceResult` eh aceitavel: cada resultado contem apenas strings pequenas (nome do evento, severidade, recomendacao) e poucos `double`. O ganho de performance vem principalmente de **nao copiar os arrays DAS/DTS** durante o processamento interno.

In [ ]:
# Demonstracao do padrao vector_to_numpy em Python puro
def vector_to_numpy_like(vec):
    """Move os dados para um objeto que mantem o buffer vivo."""
    arr = np.frombuffer(
        buffer=bytes(vec),  # em C++ seria o mesmo buffer, aqui simulamos
        dtype=np.float64,
        count=len(vec)
    )
    # Em pybind11, a capsule mantem o vetor C++ vivo.
    return arr

v = np.arange(5, dtype=np.float64)
view = vector_to_numpy_like(v)
print(view)

# Aviso: no caso real, pybind11 evita essa conversao para bytes.
# O array NumPy aponta para os mesmos bytes do std::vector original.

## 5. Ganho de performance e seguranca para o Alakoro

### 5.1 Evita copiar arrays entre regras

As regras de inferencia operam sobre perfis temporarios:

```cpp
auto mean_profile = detail::temporal_mean(dts, n_times, n_channels);
auto anomaly = detail::remove_polynomial_baseline(std::move(mean_profile), 2);
```

Sem `std::move`, `remove_polynomial_baseline` receberia uma referencia constante e teria que alocar um novo vetor para o resultado. Com `std::move`, a funcao recebe o vetor por valor e pode reutilizar seu buffer para armazenar o resultado, economizando uma alocacao e uma copia.

### 5.2 Corrotinas sem copia

`ResultGenerator` eh **move-only**. Isso garante que:

- cada frame de corrotina tenha exatamente um dono;
- nao haja double-free ou uso apos destruicao;
- o `std::coroutine_handle` seja destruido exatamente uma vez no destrutor.

### 5.3 Composicao segura de resultados

`std::make_move_iterator` permite fundir resultados parciais de varias regras no vetor final sem copiar strings:

```cpp
out.insert(out.end(),
           std::make_move_iterator(partial.begin()),
           std::make_move_iterator(partial.end()));
```

### 5.4 Estimativa de impacto

Para uma aquisicao tipica de DAS/DTS com $N_t \times N_c$ amostras:

- `temporal_mean` produz um vetor de $N_c$ doubles.
- `remove_polynomial_baseline` produz outro vetor de $N_c$ doubles.

Sem move: cada etapa aloca um novo buffer e copia $N_c$ elementos.  
Com move: a segunda etapa reusa o buffer da primeira. Em $N_c = 10.000$, isso evita copiar ~80 KB por etapa, por regra, por execucao. Multiplicado pelas 15 regras canonicas e por muitas iteracoes de processamento, o ganho se torna mensuravel em tempo e pressao sobre o garbage collector / allocator.

In [ ]:
# Estimativa pratica de economia de copia
import numpy as np

n_channels = 10_000
n_rules = 15
n_executions = 1_000
bytes_per_double = 8

saved_per_step = n_channels * bytes_per_double  # 80 KB
saved_total = saved_per_step * n_rules * n_executions

print(f"Dados economizados por etapa move: {saved_per_step/1024:.1f} KB")
print(f"Economia total ({n_rules} regras x {n_executions} execucoes): {saved_total/(1024**2):.1f} MB")

# Em C++ a economia eh real porque evitamos alocacoes; em Python o GC/allocator
# gerencia isso, mas o principio de reuso de buffers eh analogo ao NumPy in-place.

## 6. Armadilhas

### 6.1 Usar um objeto depois de `std::move`

```cpp
std::vector<double> a = {1.0, 2.0, 3.0};
std::vector<double> b = std::move(a);
std::cout << a.size();  // valido? tecnicamente sim, mas o valor eh indefinido
a.push_back(4.0);       // valido — a esta em estado valido, mas vazio
```

A unica garantia apos `std::move(a)` eh que `a` esta em um estado valido para destruicao ou reassinalacao. Ler seu tamanho ou conteudo sem reassinalar eh um bug silencioso.

No Alakoro, evite:

```cpp
auto gen = SomeRule::apply(...);
auto results = collect_results(std::move(gen));
gen.resume();  // ERRO: gen nao possui mais o handle
```

### 6.2 Self-assignment no move assignment

```cpp
Buffer& Buffer::operator=(Buffer&& other) noexcept {
    if (this != &other) {  // guarda essencial
        data_ = std::move(other.data_);
    }
    return *this;
}
```

Sem a verificacao `this != &other`, `b = std::move(b)` poderia destruir o proprio recurso antes de move-lo. O `ResultGenerator` do Alakoro implementa essa guarda corretamente:

```cpp
ResultGenerator& operator=(ResultGenerator&& other) noexcept {
    if (this != &other) {
        if (handle_) handle_.destroy();
        handle_ = other.handle_;
        other.handle_ = nullptr;
    }
    return *this;
}
```

### 6.3 Excecoes em move constructors

Se um move constructor nao for `noexcept`, a biblioteca padrao pode recuar para copias em operacoes como `std::vector::resize` ou `std::vector::push_back`. Por isso, recursos gerenciados manualmente devem ter move constructors marcados `noexcept`.

No Alakoro:

```cpp
ResultGenerator(ResultGenerator&& other) noexcept : handle_(other.handle_) {
    other.handle_ = nullptr;
}
```

A operacao eh apenas trocar dois ponteiros; nao ha alocacao, portanto `noexcept` eh apropriado.

### 6.4 `std::move` em objetos que ja sao rvalues

```cpp
return std::move(local);  // geralmente piora: impede RVO/NRVO
```

O compilador pode otimizar o retorno por valor sem construir o objeto local. `std::move` nesse caso forca uma chamada ao move constructor e pode eliminar a otimizacao. Prefira:

```cpp
return local;
```

No Alakoro, `make_result` retorna por valor sem `std::move`, permitindo RVO:

```cpp
template <CanonicalEvent E>
InferenceResult make_result(double confidence, double depth_md,
                            std::string_view severity) {
    return InferenceResult{
        std::string(EventTraits<E>::code),
        std::string(EventTraits<E>::label_pt),
        std::string(EventTraits<E>::label_en),
        confidence,
        depth_md,
        std::string(severity),
        std::string(EventTraits<E>::recommendation)
    };
}
```

In [ ]:
# Python: armadilha analoga — usar objeto "movido" (del/reassign)
a = [1, 2, 3]
b = a
del a
print("a foi deletado; b ainda valido:", b)

# Em C++, std::move(a) NAO deleta a; deixa em estado valido mas indefinido.
# O bug tipico eh ler a.size() sem reassinalar a.

# Demonstracao de self-assignment seguro em Python
class SafeBox:
    def __init__(self, data):
        self.data = data

    def move_from(self, other):
        if self is not other:       # equivalente a if (this != &other)
            self.data = other.data
            other.data = []         # deixa other em estado valido mas vazio
        return self

x = SafeBox([1, 2, 3])
y = SafeBox([4, 5])
y.move_from(x)
print("y:", y.data, "x:", x.data)

# Auto-atribuicao segura
z = SafeBox([7, 8, 9])
z.move_from(z)
print("z:", z.data)  # permanece [7, 8, 9]

## 7. Comparacao com Python

Python nao tem uma nocao explicita de "mover" um objeto. Variaveis Python sao **referencias** (ponteiros) para objetos alocados no heap.

```python
a = [1.0, 2.0, 3.0]
b = a          # b aponta para o mesmo objeto; nenhuma copia ocorre
a.append(4.0)  # b tambem ve o 4.0
```

Se voce quer criar uma copia independente, deve faze-lo explicitamente:

```python
import copy
b = copy.deepcopy(a)
```

Para dados NumPy, a distincao eh semelhante:

```python
import numpy as np
x = np.array([1.0, 2.0, 3.0])
y = x              # view/referencia, zero-copy
z = x.copy()       # copia explicita
```

Em C++:

```cpp
std::vector<double> a = {1.0, 2.0, 3.0};
std::vector<double> b = a;            // copia explicita (como deepcopy)
std::vector<double> c = std::move(a); // transferencia de posse
```

A semantica de movimento em C++ eh, portanto, uma ferramenta de **controle de lifetime e performance** que nao tem equivalente direto em Python. No binding pybind11, o `vector_to_numpy` aproxima-se do ideal Python: move o `std::vector` para uma capsula e entrega uma view NumPy zero-copy.

In [ ]:
# Tabela comparativa Python vs C++
import pandas as pd

comparison = pd.DataFrame({
    "Operacao": [
        "Atribuicao simples",
        "Copiar estrutura",
        "Transferir posse (move)",
        "Objeto unico nao-copiavel",
        "Retorno zero-copy para NumPy"
    ],
    "Python": [
        "b = a  (referencia)",
        "copy.deepcopy(a)",
        "del a; b = a (manual, nao semantico)",
        "Gerador, arquivo aberto, lock",
        "np.array(obj, copy=False) ou capsule"
    ],
    "C++": [
        "T& b = a  (referencia)",
        "T b = a;  // copy ctor",
        "T b = std::move(a);",
        "= delete no copy ctor; move ctor manual",
        "py::capsule + vector_to_numpy"
    ]
})
print(comparison.to_string(index=False))

## 8. Resumo

| Conceito | O que faz | Exemplo no Alakoro |
|----------|-----------|-------------------|
| `T&&` | Referencia para rvalue; permite roubar recursos | `ResultGenerator(ResultGenerator&&)` |
| `std::move` | Converte lvalue em rvalue | `collect_results(std::move(gen))` |
| Move constructor | Transfere posse sem copiar | `ResultGenerator` move-only |
| Move assignment | Substitui o objeto atual roubando recursos | `ResultGenerator::operator=(ResultGenerator&&)` |
| `= delete` | Impede copia de recursos unicos | `ResultGenerator(const ResultGenerator&) = delete` |
| `std::make_move_iterator` | Move elementos entre containers | `out.insert(..., make_move_iterator(...))` |
| `noexcept` | Garante que move nao lance excecoes | Todos os moves de `ResultGenerator` |

A semantica de movimento eh um dos pilares de performance do motor de inferencia C++20 do Alakoro. Ela permite processar arrays DAS/DTS de grande volume sem copias desnecessarias, mantendo ao mesmo tempo seguranca de lifetime atraves de tipos move-only como `ResultGenerator`.

## Referencias

- `src/cpp/include/alakoro/inference_engine.hpp` — definicao de `ResultGenerator`, `InferenceEngine`, `collect_results` e regras canonicas.
- `src/cpp/src/bindings.cpp` — bindings pybind11, `vector_to_numpy` e exposicao de `CanonicalInferenceEngine`.
- C++ Standard: `[class.copy]`, `[class.copy.assign]`, `[expr.prim.lambda.capture]`, `[coroutine.handle]`.